# Feature Importance Analysis with Integrated Gradients

In [1]:
from pydoc import locate

from captum.attr import IntegratedGradients
import lightning as L
import matplotlib.pyplot as plt
import torch
from torch import Tensor
import wandb
import xarray as xr

from deeprec.data import DeepRecDataModule
from deeprec.utils import ROOT_DIR, wandb_checkpoint_download

## Load model weights

In [2]:
wandb_project = "deeprec-paper_ensembles"
run_id = "6pz8pwa4"
alias = "best"
# Download checkpoint
ckpt_file = wandb_checkpoint_download(project=wandb_project, run_id=run_id, alias=alias)
# Download model config
api = wandb.Api()
run = api.run(f"{wandb_project}/{run_id}")
config = run.config

wandb:   1 of 1 files downloaded.  


In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"{device = }")

device = device(type='mps')


In [4]:
# Model creation
model_class = locate(config["model"]["class_path"])
if issubclass(model_class, L.LightningModule):
    model = model_class.load_from_checkpoint(ckpt_file, **config["model"])
else:
    raise TypeError(f"Provided class_path '{model_class}' is not a LightningModule.")
model.eval()
model.to(device);

In [5]:
class ModelWrapper(L.LightningModule):
    """Wraps the model to accept tuples instead of a dictionary.

    Only returns TWSA (mu), not the uncertainty (b)."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, xt: Tensor, xm: Tensor, xv: Tensor):
        # Manually set requires_grad to True as we do not use PyTorch Lightning
        xt = xt.requires_grad_()
        xm = xm.requires_grad_()
        xv = xv.requires_grad_()
        print(f"{xt.dtype = }")
        print(f"{xm.dtype = }")
        print(f"{xv.dtype = }")
        # forward pass
        out = self.model(tensor_inputs=xt, matrix_inputs=xm, vector_inputs=xv)

        # Only return the mean (first output)
        mu_idx = 0
        return out[:, mu_idx]

In [6]:
wrapped = ModelWrapper(model)

## Explore Captum


In [7]:
# Data creation
dm = DeepRecDataModule(**config["data"])
# Get inputs during train time (range for baseline calc)
times_train = dm._time_train
start, end = times_train[0], times_train[-1]
inputs = dm._inputs.sel(time=slice(start, end))

# Get tensors for the unchanged inputs
# We use the predict stage as it leaves inputs in original order
dm._inputs = inputs
dm.setup("predict")
dl_inputs = dm.predict_dataloader()

DataModule: Preparing predict tensors...


In [8]:
# Define climatology and static baseline inputs
inputs_clim = [
    "era5_d2m",
    "era5_t2m",
    "era5_e",
    "era5_lai_hv",
    "era5_lai_lv",
    "era5_pev",
    "era5_sro",
    "era5_ssro",
    "era5_sp",
    "era5_tp",
    "era5_swvl1",
    "era5_swvl2",
    "era5_swvl3",
    "era5_swvl4",
]
inputs_static = [
    "nvec_x",
    "nvec_y",
    "nvec_z",
    "cell_area",
    "cropland_irrigated",
    "cropland_rainfed",
    "pastures",
    "urbanareas",
    "lakes",
    "year_sin",
    "year_cos",
    "year2_sin",
    "year2_cos",
    "year2_cos",
    "oni",
]


In [9]:
# Calculate baseline for all inputs over train period within area mask
mask = dm._mask
inputs_baseline = inputs.copy(deep=True)
# Month selector with time dimension
time_index = inputs.get_index("time")
months = time_index.month
selector = xr.DataArray(months, coords=[time_index])
# Replace inputs with their per-grid-cell monthly climatology
for inp in inputs_clim:
    da = inputs_baseline[inp]
    # Dims: (month, lat, lon)
    clim = da.groupby("time.month").mean(dim=["time"], skipna=True)
    # Dims: (time, lat, lom)
    clim_time = clim.sel(month=selector).drop_vars("month")
    inputs_baseline[inp] = clim_time
    # inputs_baseline[inp] = xr.zeros_like(da)
# Replace inputs with their per-grid-cell mean
for inp in inputs_static:
    da = inputs_baseline[inp]
    if "lat" in da.dims or "lon" in da.dims:
        da = da.where(mask == 1)
    mean = da.mean().broadcast_like(inputs_baseline[inp])
    inputs_baseline[inp] = mean


In [10]:
inputs_baseline

<xarray.Dataset> Size: 5GB
Dimensions:             (time: 237, lat: 360, lon: 720)
Coordinates:
  * lat                 (lat) float32 1kB 89.75 89.25 88.75 ... -89.25 -89.75
  * lon                 (lon) float32 3kB -179.8 -179.2 -178.8 ... 179.2 179.8
  * time                (time) datetime64[ns] 2kB 2002-04-01 ... 2021-12-01
Data variables: (12/28)
    era5_d2m            (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    era5_t2m            (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    era5_e              (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    era5_lai_hv         (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    era5_lai_lv         (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    era5_pev            (time, lat, lon) float32 246MB dask.array<chunksize=(12, 120, 120), meta=np.ndarray>
    ...                  ...
    lakes               (time, lat, lon) float32 246MB dask.array<chunksize=(237, 360, 720), meta=np.ndarray>
    year_sin            (time) float32 948B dask.array<chunksize=(237,), meta=np.ndarray>
    year_cos            (time) float32 948B dask.array<chunksize=(237,), meta=np.ndarray>
    year2_sin           (time) float32 948B dask.array<chunksize=(237,), meta=np.ndarray>
    year2_cos           (time) float32 948B dask.array<chunksize=(237,), meta=np.ndarray>
    oni                 (time) float32 948B dask.array<chunksize=(237,), meta=np.ndarray>

In [11]:
# Create a baseline dataloader
dm._inputs = inputs_baseline
dm.setup("predict")
dl_baseline = dm.predict_dataloader()

DataModule: Preparing predict tensors...


In [12]:
assert len(dl_inputs) == len(dl_baseline)

In [13]:
def dict_to_tuple(dict_: dict[str, Tensor]) -> tuple[Tensor, Tensor, Tensor]:
    input_order = ("tensor_inputs", "matrix_inputs", "vector_inputs")
    return tuple(dict_[inp] for inp in input_order)

In [14]:
b_input = dict_to_tuple(next(iter(dl_inputs))["inputs"])
b_baseline = dict_to_tuple(next(iter(dl_baseline))["inputs"])

In [16]:
for t in b_input:
    print(t.shape)

torch.Size([256, 14, 13, 35, 35])
torch.Size([256, 9, 35, 35])
torch.Size([256, 5, 13])


In [17]:
b_input_small = tuple(t[:10].to(device) for t in b_input)
b_baseline_small = tuple(t[:10].to(device) for t in b_baseline)

In [18]:
with torch.no_grad():
    y = wrapped(*b_input_small)
y

xt.dtype = torch.float32
xm.dtype = torch.float32
xv.dtype = torch.float32


tensor([-87.2801, -90.1890, -19.0829, -38.2625, -43.3399, -49.1305, -56.0894,
        -54.7567, -62.7597, -70.4296], device='mps:0')

In [19]:
for t in b_input_small:
    print(f"{t.dtype = }")
for t in b_baseline_small:
    print(f"{t.dtype = }")

t.dtype = torch.float32
t.dtype = torch.float32
t.dtype = torch.float32
t.dtype = torch.float32
t.dtype = torch.float32
t.dtype = torch.float32


In [ ]:
ig = IntegratedGradients(wrapped)
# attributions, approximation_error = ig.attribute(
attributions = ig.attribute(
    b_input_small,
    baselines=b_baseline_small,
    method="gausslegendre",
    n_steps=20,
    # return_convergence_delta=True,
)

xt.dtype = torch.float32
xm.dtype = torch.float32
xv.dtype = torch.float32


TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.

In [ ]:
att_t, att_m, att_v = attributions

In [ ]:
# Sum attributions over temporal and spatial dimensions
# Sum over time, lat, lon
att_t_mean = att_t.sum(dim=[-3, -2, -1])
# Sum over lat, lon
att_m_mean = att_m.sum(dim=[-2, -1])
# Sum over time
att_v_mean = att_v.sum(dim=-1)
print(att_t_mean.shape)
print(att_m_mean.shape)
print(att_v_mean.shape)

torch.Size([1, 14])
torch.Size([1, 9])
torch.Size([1, 5])


In [ ]:
# Mean over batch dimension
for name, att in zip(dm._tensor_input_vars, att_t_mean.mean(0)):
    print(f"{name:20}: {att: .4f}")
for name, att in zip(dm._matrix_input_vars, att_m_mean.mean(0)):
    print(f"{name:20}: {att: .4f}")
for name, att in zip(dm._vector_input_vars, att_v_mean.mean(0)):
    print(f"{name:20}: {att: .4f}")

era5_d2m            : -0.3479
era5_t2m            : -0.3913
era5_e              :  0.4738
era5_lai_hv         : -0.0000
era5_lai_lv         : -0.0000
era5_pev            : -0.0287
era5_sro            :  0.0401
era5_ssro           : -1.0067
era5_sp             :  0.1200
era5_tp             : -0.8854
era5_swvl1          : -1.3715
era5_swvl2          :  0.2587
era5_swvl3          : -2.0960
era5_swvl4          :  0.8068
nvec_x              : -54.4722
nvec_y              :  1.4284
nvec_z              :  1.5518
cell_area           : -0.3363
cropland_irrigated  : -1.0718
cropland_rainfed    :  22.2122
pastures            : -35.8618
urbanareas          : -165.2464
lakes               :  4.0342
year_sin            :  0.8121
year_cos            :  1.0896
year2_sin           :  0.2055
year2_cos           :  0.9119
oni                 : -0.7284
